In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError(
        "GOOGLE_API_KEY is missing. Add it to the project .env file before running this notebook."
    )

# Section 1: What Are Guardrails?

> **Guardrails are safety checks for AI applications.** They inspect information as it moves through an agent and prevent unsafe, invalid, or unwanted behavior before it reaches the user or a real-world tool.

## Where They Intercept an Agent

```mermaid
flowchart LR
    A[User input] --> B[Input guardrail]
    B --> C[Model and tools]
    C --> D[Output guardrail]
    D --> E[Safe response]

    B -. blocks or cleans .-> X[Unsafe request]
    D -. blocks or revises .-> Y[Unsafe response]
```

Guardrails can run at three useful points:

- **Before the agent starts:** validate or sanitize the user's input.
- **Around model and tool calls:** control what the agent is allowed to ask for or execute.
- **After the agent completes:** check the final response before showing it to the user.

## Common Use Cases

| Goal | What the guardrail does | Example |
| --- | --- | --- |
| **Protect private data** | Detects, masks, or blocks sensitive information | Redact email addresses or credit-card numbers before logging |
| **Stop prompt injection** | Detects instructions that try to bypass the agent's rules | Block a request to ignore system instructions |
| **Filter harmful content** | Rejects unsafe requests or responses | Prevent dangerous instructions from being returned |
| **Enforce business rules** | Requires checks or approval before sensitive actions | Ask for approval before a financial operation |
| **Improve output quality** | Confirms that the response follows a required format | Ensure an answer includes valid JSON or required fields |

## The Core Idea

A guardrail should answer one simple question at each boundary:

> **Is this input, action, or output safe and valid for the next step?**

If the answer is **no**, the guardrail can stop the run, clean the content, ask for clarification, or route the case for human review.

# Section 2: Choosing the Right Guardrail

There are two main ways to validate an agent's behavior. The best choice depends on whether the rule is **easy to state exactly** or requires **context and judgment**.

## Two Complementary Approaches

| Approach | How it works | Strengths | Limitations | Best for |
| --- | --- | --- | --- | --- |
| **Deterministic** | Uses explicit rules such as regex, keyword matching, schemas, or programmatic checks | Fast, predictable, inexpensive, and easy to test | Can miss meaning, context, and indirect wording | PII detection, required fields, allowed values, and hard business rules |
| **Model-based** | Uses an LLM or classifier to evaluate meaning and context | Understands nuance, paraphrases, and intent | Slower, more expensive, and less perfectly predictable | Prompt injection, harmful intent, tone, and policy classification |

## A Simple Example

**Deterministic check:**

> “Does this response contain an email address or credit-card number?”

This can usually be answered with a regular expression or a specialized detector. The expected result is clear and repeatable.

**Model-based check:**

> “Does this request attempt to bypass the agent's safety rules?”

This requires understanding intent. The same idea can be expressed in many different ways, so a model or classifier is often better suited to the task.

## Which One Should You Use?

- Choose **deterministic guardrails** when the rule has a precise, testable definition.
- Choose **model-based guardrails** when the decision depends on context, intent, or meaning.
- Use **both** when the workflow is important: let deterministic checks handle obvious violations, then use a model-based check for nuanced cases.

> **Practical pattern:** use the fastest reliable check first, and reserve model-based evaluation for the cases that need judgment.

This layered approach keeps common requests fast while adding deeper review where it provides the most value.

In [5]:
import re

# --- Deterministic approach ---
def deterministic_guardrail(text: str) -> bool:
    """Returns True if content is blocked."""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]
    return any(kw in text.lower() for kw in banned_keywords)

test_inputs = [
    "How do I hack into a database?",
    "What is the capital of France?",
    "Explain how malware spreads",
]

print("=== Deterministic Guardrail Demo ===")
for inp in test_inputs:
    blocked = deterministic_guardrail(inp)
    status = "🚫 BLOCKED" if blocked else "✅ ALLOWED"
    print(f"{status}: {inp}")

=== Deterministic Guardrail Demo ===
🚫 BLOCKED: How do I hack into a database?
✅ ALLOWED: What is the capital of France?
🚫 BLOCKED: Explain how malware spreads


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

# --- Model-based approach ---
model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0,
)


def model_based_guardrail(text: str) -> str:
    """Use an LLM to classify content as SAFE or UNSAFE."""
    prompt = f"""Classify the following user input for safety.
Reply with exactly one word: SAFE or UNSAFE.

Input: {text}"""
    result = model.invoke([{"role": "user", "content": prompt}])
    content = result.content

    if isinstance(content, list):
        content = "".join(
            block.get("text", "")
            for block in content
            if isinstance(block, dict)
        )

    verdict = content.strip().upper()
    if verdict not in {"SAFE", "UNSAFE"}:
        return "UNSAFE"
    return verdict


print("=== Model-Based Guardrail Demo ===")
for inp in test_inputs:
    verdict = model_based_guardrail(inp)
    status = "BLOCKED" if verdict == "UNSAFE" else "ALLOWED"
    print(f"[{status}] {inp}")

=== Model-Based Guardrail Demo ===
[BLOCKED] How do I hack into a database?
[ALLOWED] What is the capital of France?


## Section 3: Built-in Guardrail: PII Detection Middleware

LangChain includes `PIIMiddleware` for detecting and handling **personally identifiable information (PII)** as it moves through an agent. This helps prevent sensitive data from being exposed in prompts, tool calls, logs, or final responses.

### Supported PII Types

| PII type | Example |
| --- | --- |
| `email` | `user@example.com` |
| `credit_card` | `5105-1051-0510-5100` |
| `ip` | `192.168.1.1` |
| `mac_address` | `00:1A:2B:3C:4D:5E` |
| `url` | `https://secret-site.com` |

### Handling Strategies

| Strategy | Result | Best used when |
| --- | --- | --- |
| `redact` | Replaces the value with a label such as `[REDACTED_EMAIL]` | The information should be removed but its presence should remain clear |
| `mask` | Hides part of the value, such as `****-****-****-1234` | Users need to recognize the value without seeing it in full |
| `hash` | Replaces the value with a one-way digest such as `a8f5f167...` | The value needs to be matched consistently without revealing the original |
| `block` | Raises an exception and stops the operation | The presence of PII must prevent the request from continuing |

> **Choosing a strategy:** use `redact` for safe display and logging, `mask` for partial visibility, `hash` for consistent comparisons, and `block` for high-risk workflows.

A production agent should apply PII middleware at the boundary where sensitive data first appears, then verify that the selected strategy matches the privacy requirements of the application.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

# Define a simple dummy tool
@tool
def customer_lookup(query: str) -> str:
    """Look up customer information."""
    return f"Customer record found for query: {query}"

# Create agent with PII Middleware
agent = create_agent(
    model="gpt-4o",
    tools=[customer_lookup],
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

print("Agent with PII middleware created successfully!")

In [ ]:
# Test PII Redaction
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "My email is john.doe@example.com and my card is 5105-1051-0510-5100. Can you help me?"
    }]
})

print("=== Agent Response ===")
print(result["messages"][-1].content)